# 01 · Environment setup & single-image classification

This notebook walks the full **first step** of the capstone pipeline:

1. Configure OpenRouter + Braintrust credentials.
2. Load a document image (from your local RVL-CDIP tree, or fetched from the
   committed Braintrust slice).
3. Normalize it exactly like the dataset slices (grayscale, 1024x1024, white padding).
4. Ask an OpenRouter vision model to classify it, then parse the prediction.

Everything reuses the repo's `src/` library rather than reimplementing logic.
See `scripts/braintrust/braintrust_openrouter_input.py` for the full eval runner.


## Prerequisites

```bash
pip install -r requirements-dev.txt
```

Two env files (both gitignored) must exist:

- `.env` — `OPENROUTER_API_KEY` (and optionally `RESEARCH_FUNDING_API_KEY`).
- `braintrust.env` — single source of truth for the Braintrust org/project/dataset/model.

Create them from the templates:

```bash
cp .env.example .env
cp braintrust.env.example braintrust.env
```

`src/braintrust_config.py` loads `braintrust.env` **first** and only falls back to
`.env`, so the Braintrust key always resolves to the current account. Always read
keys/ids through `config` from `load_braintrust_config()` — never straight from
`os.environ` — or a stale `.env` value can silently win.


## 0. Bootstrap: repo path + credentials

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


In [ ]:
from src.braintrust_config import load_braintrust_config
from src.env_utils import require_env

config = load_braintrust_config()      # braintrust.env first, then .env
api_key = require_env("OPENROUTER_API_KEY")[0]

print("project:", config.project_name)
print("project_id:", config.project_id)
print("dataset:", config.dataset_project, "/", config.dataset)
print("model:", config.model)
print("braintrust api_key set:", bool(config.api_key))
print("openrouter api_key set:", bool(api_key))


## 1. Load a document image

Images follow the filename convention `rvl_cdip__{class}__{NNNN}.png`, which embeds the
ground-truth class so `extract_class_from_filename()` can recover the label. If you have a
local RVL-CDIP tree (`processed_balanced_dataset/images`, `fixed_size_sampled`, a Kaggle
download, ...) this notebook uses it. Otherwise it fetches **one real image** from the
committed Braintrust slice `fixed_size_sampled`, so the demo always works out of the box.

In [ ]:
from src.image_utils import find_images

# EDIT ME: point at your local RVL-CDIP tree if you have one.
image_dir = ROOT / "processed_balanced_dataset" / "images"
paths = find_images(image_dir, recursive=True) if image_dir.exists() else []
print(f"{len(paths)} images found under {image_dir}")

if not paths:
    print("No local tree found - fetching one real image from the committed "
          f"Braintrust slice '{config.dataset}' instead.")
    import tempfile

    import braintrust

    from src.braintrust_utils import fetch_attachment_bytes

    braintrust.login(api_key=config.api_key)
    ds = braintrust.init_dataset(project=config.project_name, name=config.dataset)
    row = next(
        r for r in ds
        if not ((r.get("input") or {}).get("metadata") or {}).get("placeholder", False)
        and (r.get("input") or {}).get("image") is not None
    )
    att = (row["input"] or {})["image"]
    raw = fetch_attachment_bytes(config.api_key, att.reference, config.org_id, config.api_base)
    expected = row.get("expected") or "unknown"
    _tmp = Path(tempfile.mkdtemp(prefix="rvl_cdip_nb01_"))
    image_path = _tmp / f"rvl_cdip__{expected}__0001.png"
    image_path.write_bytes(raw)
    print("Fetched:", att.reference.get("filename", image_path.name),
          f"({len(raw):,} bytes)")

image_path = paths[0] if paths else image_path
print("Using:", image_path)


In [ ]:
from IPython.display import Image as IPImage, display


def extract_class_from_filename(filename: str) -> str:
    # Recover the ground-truth class from 'rvl_cdip__{class}__{NNNN}.png'.
    parts = filename.split("__")
    return parts[1] if len(parts) >= 3 else "?"


display(IPImage(filename=str(image_path), width=360))
print("Filename:", image_path.name)
expected = extract_class_from_filename(image_path.name)
print("Ground-truth class:", expected)


## 2. Normalize to the standard representation

Every dataset slice stores **grayscale 1024x1024 PNGs with white padding** that
preserves the aspect ratio (`src/image_utils.resize_with_padding`). Normalizing a
single image the same way means a standalone classification matches what the eval
runner would see.

In [ ]:
from PIL import Image

from src.image_utils import resize_with_padding

img = Image.open(image_path).convert("L")
normalized = resize_with_padding(img, (1024, 1024), fill=255)
print("Normalized size:", normalized.size, "| mode:", normalized.mode)
display(normalized)


## 3. Build the request payload

`src/openrouter_utils.build_vision_messages` packs the prompt text + base64 image into
an OpenAI-style `messages` payload. Use the repo's current default prompt
(`get_prompt(DEFAULT_PROMPT_VERSION)` = v17.2) rather than a hardcoded string.

In [ ]:
from src.image_utils import encode_image_base64
from src.openrouter_utils import build_vision_messages
from src.prompts import DEFAULT_PROMPT_VERSION, get_prompt

prompt = get_prompt(DEFAULT_PROMPT_VERSION)
image_b64 = encode_image_base64(image_path)
messages = build_vision_messages(prompt, image_b64, image_format="png")

print("Prompt version:", DEFAULT_PROMPT_VERSION, "| chars:", len(prompt))
print("Message role:", messages[0]["role"], "| content parts:", len(messages[0]["content"]))


## 4. Classify (spends OpenRouter credits)

The one-liner `classify_image()` uses the module's pinned prompt (v14). To classify
with the **current default prompt** (v17.2), send the payload built above directly and
parse the answer with `clean_prediction()` / `extract_runner_up()`.

> This is the **only** cell in the notebook suite that spends model credits — roughly
> $0.0004 per image at qwen3.7-flash rates. Everything else in the four notebooks is
> read-only or print-only.

In [ ]:
from src.openrouter_classifier import classify_image

result = classify_image(api_key, image_path, model=config.model)
print("model:", result["model"])
print("classification:", result["classification"])
print("status:", result["status"])
print("usage:", result.get("usage"))
print("exact_match:", result["classification"] == expected)


In [ ]:
import requests

from src.openrouter_classifier import clean_prediction, extract_runner_up
from src.openrouter_utils import OPENROUTER_API_URL

payload = {
    "model": config.model,
    "messages": build_vision_messages(prompt, image_b64),
    "max_tokens": 4096,
    "temperature": 0.1,
}
resp = requests.post(
    OPENROUTER_API_URL,
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json=payload,
    timeout=120,
)
resp.raise_for_status()
data = resp.json()
raw = data["choices"][0]["message"].get("content") or ""
print("raw tail:", raw[-300:])
print("clean_prediction:", clean_prediction(raw))
print("runner_up:", extract_runner_up(raw))
print("usage:", data.get("usage"))


## Next

Notebook **02 · Balanced sampling & Braintrust upload** turns this single-image flow
into a deterministic, class-balanced dataset slice ready to be uploaded to Braintrust
and queued as a full eval run.